# Layout-Aware & Multimodal Document Parsing Demo

**References:**
- Slide 19 (*Hands-On RAG with AWS*)
- Slide 20 (*Building Agentic Workflows with RAG on AWS Bedrock*)

### Why Layout-Aware Parsing Matters
Standard text extractors flatten document pages into unformatted text strings. When naive chunking splits text at arbitrary character or token boundaries, **tables are fractured**, column alignments are severed, and chart annotations are detached from their data points.

In this demo, we:
1. Use **Claude 3.5 Sonnet / Claude 3 Haiku Multimodal Converse API** (or Bedrock Data Automation) to transcribe complex pages with tables and architecture diagrams into structured GitHub-Flavored Markdown.
2. Compare **Naive Fixed-Size Chunking** vs. **Layout-Aware Markdown Chunking**.
3. Quantify **Table Integrity** to prove why preserving semantic blocks improves RAG context quality.

In [ ]:
from multimodal_layout_aware_parsing import (
    MultimodalDocumentParser,
    NaiveFixedSizeChunker,
    LayoutAwareMarkdownChunker,
    evaluate_table_integrity
)

# Initialize parser (auto-detects credentials or falls back to mock mode)
parser = MultimodalDocumentParser(region_name="us-east-1")
print(f"Multimodal Parser initialized. Mock Mode: {parser.mock_mode}")

## 1. Extract Structured Markdown from Complex Visual Layout
We pass the visual page bytes to Claude's multimodal endpoint.

In [ ]:
# In production, load actual image: with open('report_page.png', 'rb') as f: img_bytes = f.read()
markdown_document = parser.parse_image_bytes(b"sample_image_bytes")
print("Extracted Markdown Document:\n")
print(markdown_document)

## 2. Naive Fixed-Size Chunking (The Problem)
Fixed-size chunkers split strictly by character count with overlap, cutting across table rows and severing table headers.

In [ ]:
naive_chunker = NaiveFixedSizeChunker(chunk_size=300, chunk_overlap=50)
naive_chunks = naive_chunker.chunk(markdown_document)
naive_metrics = evaluate_table_integrity(naive_chunks)

print(f"Total Chunks Produced: {len(naive_chunks)}")
print(f"Table Integrity Score: {naive_metrics['table_integrity_score']:.1%}")
print(f"Fractured Table Chunks: {naive_metrics['fractured_tables']}")

print("\n--- Sample Fractured Chunk (Missing Table Header & Separator) ---")
for ch in naive_chunks:
    if '|' in ch['content'] and '|---|' not in ch['content']:
        print(ch['content'])
        break

## 3. Layout-Aware Markdown Chunking (The Solution)
Layout-aware chunking treats tables, diagrams, and section blocks as atomic units.

In [ ]:
layout_chunker = LayoutAwareMarkdownChunker(target_chunk_size=600)
layout_chunks = layout_chunker.chunk(markdown_document)
layout_metrics = evaluate_table_integrity(layout_chunks)

print(f"Total Chunks Produced: {len(layout_chunks)}")
print(f"Table Integrity Score: {layout_metrics['table_integrity_score']:.1%}")
print(f"Fractured Table Chunks: {layout_metrics['fractured_tables']}")

print("\n--- Preserved Table Chunk (Complete Structure Maintained) ---")
for ch in layout_chunks:
    if ch['contains_table']:
        print(ch['content'])
        break

## 4. Key Takeaways for Advanced RAG
- Multimodal parsing converts unstructured PDF/image graphics into standardized Markdown with explicit tables.
- Chunking must respect structural boundaries to prevent retrieval degradation when answering questions about tabular metrics or architectural flows.